# M8c · Data Encoding

**Goal:** convert categorical columns to numeric form so that estimators
(linear models, XGBoost, neural nets) can use them. Three encoding
families covered, each appropriate for a different categorical pattern.

| Encoding                            | When to use                                             | Trade-off                            |
|-------------------------------------|---------------------------------------------------------|--------------------------------------|
| **One-Hot (Nominal)**               | Few categories with no order (e.g. `vehicle_make`).     | Explodes column count for high-card. |
| **Label / Ordinal**                 | Categories with a *natural order* (e.g. severity).      | Imposes order on linear models.      |
| **Target-Guided Ordinal**           | High-cardinality nominal w/ predictive target.          | Uses target — risks leakage.         |

**Exit criteria:**
- Encoded DataFrame with no `object`/`category` columns remaining.
- All three techniques demonstrated on a real column.
- Saved to `data/ml/step3_encoded.parquet`.


In [3]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — winsorized features + raw categoricals from dim tables

In [4]:
import pandas as pd, pathlib
df = pd.read_parquet('../../data/ml/step2_winsorized.parquet')

# Pull the raw categorical attributes that v_ml_features_full does NOT carry
# (it integer-encodes vehicle_class for the baseline pipeline).
with get_engine().connect() as conn:
    dims = pd.read_sql(text('''
        SELECT dd.tenant_id, dd.device_id,
               COALESCE(NULLIF(dv.mark_clean, ''), NULLIF(dv.mark_raw, ''), 'unknown') AS vehicle_make,
               dv.vehicle_class
          FROM warehouse.dim_device dd
          JOIN warehouse.dim_vehicle dv ON dv.vehicle_sk = dd.vehicle_sk
    '''), conn)

df_cat = df.merge(dims, on=['tenant_id', 'device_id'], how='left')
df_cat[['vehicle_make', 'vehicle_class']] = df_cat[['vehicle_make', 'vehicle_class']].fillna('unknown')
print('shape after merge:', df_cat.shape)
df_cat[['vehicle_make','vehicle_class']].nunique()


shape after merge: (23642, 64)


vehicle_make     278
vehicle_class      3
dtype: int64

## 3. Strategy A — One-Hot Encoding (Nominal)

Use for `vehicle_make` as a nominal encoding example. If cardinality is high,
compare this with the target-guided encoding below.


In [5]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
make_ohe = ohe.fit_transform(df_cat[['vehicle_make']])
make_ohe_df = pd.DataFrame(make_ohe,
                            columns=[f'make_{c}' for c in ohe.categories_[0]],
                            index=df_cat.index)
print('OHE columns:', list(make_ohe_df.columns))
make_ohe_df.head(3)


OHE columns: ['make_1208 Tu 193 Scania', 'make_425101', 'make_425102', 'make_425108', 'make_425109', 'make_425110', 'make_425113', 'make_425114', 'make_425115', 'make_425117', 'make_425118', 'make_425119', 'make_425122', 'make_425123', 'make_425125', 'make_425126', 'make_425128', 'make_425130', 'make_425133', 'make_425138', 'make_425139', 'make_425140', 'make_425141', 'make_425144', 'make_425145', 'make_425150', 'make_425151', 'make_425155', 'make_425157', 'make_425158', 'make_425162', 'make_425164', 'make_425169', 'make_425171', 'make_425182', 'make_425187', 'make_425192', 'make_425201', 'make_425202', 'make_425203', 'make_425205', 'make_425206', 'make_425207', 'make_425208', 'make_425209', 'make_425210', 'make_425211', 'make_425212', 'make_425213', 'make_425214', 'make_425215', 'make_425221', 'make_425228', 'make_425229', 'make_425232', 'make_425233', 'make_425234', 'make_425235', 'make_425236', 'make_425237', 'make_425239', 'make_425240', 'make_425248', 'make_425249', 'make_425250',

,make_1208 Tu 193 Scania,make_425101,make_425102,make_425108,make_425109,make_425110,make_425113,make_425114,make_425115,make_425117,...,make_Mercedes-Benz,make_Nemo,make_Partner,make_Peugeot,make_Plateaux,make_Renault,make_Scania,make_Usuzu,make_Volvo,make_unknown
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Strategy B — Label / Ordinal Encoding

Use for `vehicle_class` because there's a true ordering: heavy > medium > light.
Manual mapping is more reliable than `LabelEncoder` (which alphabetizes).


In [6]:
ORDINAL_MAP = {'light': 0, 'medium': 1, 'heavy': 2, 'unknown': -1}
df_cat['vehicle_class_ord'] = df_cat['vehicle_class'].map(ORDINAL_MAP)
df_cat[['vehicle_class', 'vehicle_class_ord']].drop_duplicates()


,vehicle_class,vehicle_class_ord
0,heavy,2
1,unknown,-1
12,medium,1


## 5. Strategy C — Target-Guided Ordinal Encoding

For high-cardinality nominal columns, replace each category by the **mean
of the target** within that category, then rank-encode. Useful for
`vehicle_make` if it had hundreds of values (here it doesn't, this is
demonstrative).

> Compute on a **train split** to avoid leakage. Here we derive a synthetic
> target from `harsh_events_per_100km` (proxy for risky behaviour) for the
> demonstration only.


In [7]:
target = df_cat['harsh_events_per_100km']
mean_by_make = target.groupby(df_cat['vehicle_make']).mean().sort_values()
make_to_rank = {m: i for i, m in enumerate(mean_by_make.index)}
df_cat['vehicle_make_tge'] = df_cat['vehicle_make'].map(make_to_rank)
display(mean_by_make.rename('mean_harsh_per_100km').to_frame())
df_cat[['vehicle_make', 'vehicle_make_tge']].drop_duplicates().head()


,mean_harsh_per_100km
vehicle_make,
425108,0.000000
425102,0.000000
425110,0.000000
425109,0.000000
425141,0.000000
...,...
948863,10.320633
425213,11.142083
948856,19.777793


,vehicle_make,vehicle_make_tge
0,Scania,201
1,425253,46
2,Mercedes,216
3,425298,90
4,425209,208


## 6. Assemble final encoded matrix

In [8]:
if 'year_month' in df_cat.columns:
    ym = pd.to_datetime(df_cat['year_month'].astype(str) + '-01', errors='coerce')
    df_cat['feature_year'] = ym.dt.year.fillna(0).astype(int)
    df_cat['feature_month'] = ym.dt.month.fillna(0).astype(int)

drop_cols = ['vehicle_make', 'vehicle_class']
if 'year_month' in df_cat.columns:
    drop_cols.append('year_month')

df_enc = pd.concat([
    df_cat.drop(columns=drop_cols),
    make_ohe_df,
], axis=1)

# Confirm: no object dtype left.
obj_cols = df_enc.select_dtypes(include=['object', 'category']).columns.tolist()
print('object columns remaining:', obj_cols)
assert not obj_cols, 'still have non-numeric columns'

print('final shape:', df_enc.shape)
out = pathlib.Path('../../data/ml/step3_encoded.parquet')
df_enc.to_parquet(out, index=False)
print('saved to', out.resolve())


object columns remaining: []
final shape: (23642, 343)
saved to D:\data\ml\step3_encoded.parquet
